# Demo 05 : Agent failures

Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph-Student/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


## a · tool-errors

We distinguish an execution failure from a valid empty stock result.

In [2]:
import json
import sqlite3

from langchain.tools import tool

DB = data("demo-world/service.db")


@tool
def bus_info(fleet_no: str) -> str:
    """Look up a Veenhaven bus model and depot by its fleet number."""
    with sqlite3.connect(DB) as conn:
        row = conn.execute(
            "SELECT model, depot_id FROM buses WHERE fleet_no = ?", (fleet_no,)
        ).fetchone()
    return json.dumps(dict(zip(["model", "depot_id"], row))) if row else "Unknown bus"


@tool
def parts_for_model(model: str) -> str:
    """List spare parts, stock and storage depots for a bus model such as CE-12."""
    with sqlite3.connect(DB) as conn:
        rows = conn.execute(
            "SELECT part_no, name, stock, depot FROM parts WHERE model = ?", (model,)
        ).fetchall()
    return json.dumps([dict(zip(["part_no", "name", "stock", "depot"], row)) for row in rows])


tools = [bus_info, parts_for_model]
question = {"messages": [("human", "Which door actuator fits bus CE12-2021-0117, and is it in stock? Look up the bus first.")]}

In [3]:
from langchain.messages import AIMessage, HumanMessage, ToolCall
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel


class FakeToolModel(GenericFakeChatModel):
    def bind_tools(self, tools, **kwargs):
        return self

We inject a stock-service timeout. Without a handler the run raises. `ToolErrorMiddleware` returns selected execution errors as error messages, which the real model can use to report the unavailable service.

In [4]:
from langchain.agents import create_agent
from langchain.agents.middleware import ToolErrorMiddleware


@tool
def live_stock(part_no: str) -> str:
    """Read live stock for a spare part number."""
    raise TimeoutError("stock service unavailable")


fake = FakeToolModel(messages=iter([AIMessage("", tool_calls=[
    ToolCall(name="live_stock", args={"part_no": "CE-DA-03"}, id="stock-1")])]))
try:
    create_agent(fake, tools=[live_stock]).invoke({"messages": [HumanMessage("Check stock.")]})
except TimeoutError as exc:
    print("unhandled:", type(exc).__name__)


def on_error(exc, request):
    if isinstance(exc, TimeoutError):
        return "Stock service unavailable. Do not infer stock. Ask for a manual stock check."
    return None


safe = create_agent(llm, tools=[live_stock], middleware=[ToolErrorMiddleware(on_error)],
                    system_prompt="Use live_stock for stock. If unavailable, report that and stop.")
out = safe.invoke({"messages": [("human", "Check stock for CE-DA-03.")]}, {"recursion_limit": 16})
for message in out["messages"]:
    print(message.type, getattr(message, "status", ""), message.content)

unhandled: TimeoutError


human  Check stock for CE-DA-03.
ai  
tool error Stock service unavailable. Do not infer stock. Ask for a manual stock check.
ai  The live stock service is unavailable, so I can’t confirm stock for CE-DA-03. Please request a manual stock check.


## b · model-fallback

We fail the primary model locally, then use the course model as the fallback. This exercises the failure path without sending a request for an invalid model name.

In [5]:
from langchain.agents.middleware import ModelFallbackMiddleware


class UnavailableModel(FakeToolModel):
    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        print("primary: unavailable")
        raise ConnectionError("injected model outage")


primary = UnavailableModel(messages=iter([]))
agent = create_agent(primary, tools=tools, middleware=[ModelFallbackMiddleware(llm)],
                     system_prompt="Use the database for stock. Look up the bus before its parts.")
out = agent.invoke(question, {"recursion_limit": 16})
print("fallback answer:", out["messages"][-1].content)

primary: unavailable


primary: unavailable


primary: unavailable


fallback answer: Bus CE12-2021-0117 is a CE-12. Its door actuator is part **CE-DA-03**; it is **out of stock** (0 available at Rietpoort).


## c · call-limits

We script repeated stock calls to expose both limits. Run limits reset per invocation. Thread limits accumulate when state is persisted with a checkpointer.

In [6]:
from langchain.agents.middleware import ModelCallLimitMiddleware, ToolCallLimitMiddleware


def repeated_stock():
    return FakeToolModel(messages=iter([
        AIMessage("", tool_calls=[ToolCall(name="bus_info", args={"fleet_no": "CE12-2021-0117"}, id="1")]),
        AIMessage("", tool_calls=[ToolCall(name="bus_info", args={"fleet_no": "CE12-2021-0117"}, id="2")]),
        AIMessage("Finished.")]))


for middleware in [ModelCallLimitMiddleware(run_limit=1, exit_behavior="end"),
                   ToolCallLimitMiddleware(tool_name="bus_info", run_limit=1, exit_behavior="end")]:
    out = create_agent(repeated_stock(), tools=[bus_info], middleware=[middleware]).invoke(question)
    print("limit:", type(middleware).__name__)
    for message in out["messages"]:
        print(message.type, message.content or getattr(message, "tool_calls", []))

limit: ModelCallLimitMiddleware
human Which door actuator fits bus CE12-2021-0117, and is it in stock? Look up the bus first.
ai [{'name': 'bus_info', 'args': {'fleet_no': 'CE12-2021-0117'}, 'id': '1', 'type': 'tool_call'}]
tool {"model": "CE-12", "depot_id": "D01"}
ai Model call limits exceeded: run limit (1/1)
limit: ToolCallLimitMiddleware
human Which door actuator fits bus CE12-2021-0117, and is it in stock? Look up the bus first.
ai [{'name': 'bus_info', 'args': {'fleet_no': 'CE12-2021-0117'}, 'id': '1', 'type': 'tool_call'}]
tool {"model": "CE-12", "depot_id": "D01"}
ai [{'name': 'bus_info', 'args': {'fleet_no': 'CE12-2021-0117'}, 'id': '2', 'type': 'tool_call'}]
tool Tool call limit exceeded. Do not call 'bus_info' again.
ai 'bus_info' tool call limit reached: run limit exceeded (2/1 calls).


## Things to try

Optional : change the bus or part identifier and run the example again.